In [ ]:
from pydantic import BaseModel, Field

class Task(BaseModel):
    id: int = Field(..., description="The unique identifier of the task")
    title: str = Field(..., description="The title of the task")
    description: str = Field(..., description="A detailed description of the task")
    completed: bool = Field(False, description="Indicates whether the task is completed or not")

class Plan(BaseModel):
    name: str = Field(..., description="The name of the blog")
    tasks: list[Task] = Field(..., description="A list of tasks associated with the plan")

In [4]:
from typing import TypedDict, Annotated
import operator

class State(TypedDict):
    topic: str
    plan: Plan
    sections: Annotated[list[str], operator.add]
    final: str

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
plan_llm = llm.with_structured_output(Plan)

In [7]:
from langchain_core.messages import SystemMessage


def orchestrate(state: State):
    topic = state['topic']

    response = plan_llm.invoke([SystemMessage(content=f"Create a plan for the topic: {topic}\nDefine 5-6 Sections as separate tasks"), {"role" : "user", "content": f"Topic: {topic}"}])

    return {"plan" : response}

In [8]:
from langgraph.types import Send

def fan_out(state: State):
    plan = state['plan']
    return [Send("worker" , {"topic": state["topic"], "task": task, "plan": plan}) for task in plan.tasks]

In [9]:
def worker(payload: dict):
    topic = payload["topic"]
    task = payload["task"]
    plan = payload["plan"]

    blog_title = plan.name

    section = llm.invoke([SystemMessage(content="Write one clean MarkDown Section for the following task"), {"role" : "user", "content": f"""Blog Title: {blog_title}\nTopic: {topic}\nSection: {task.title}\nDescription: {task.description} 
    Return only the section content in MarkDown format without any additional text or explanation."""}])

    return {"result": section.text.strip()}